# Gold ao vivo — `gold_orders_live`

Mantém o **estado atual de cada pedido** a partir do feed de eventos de status da bronze, tolerando eventos duplicados e fora de ordem. Desenho: `docs/design/fase-5-streaming.md`; decisões: ADRs 0009 a 0012.

| Peça | Decisão |
|---|---|
| Execução | Structured Streaming com `foreachBatch`, gatilho `AvailableNow` (o serverless não suporta `ProcessingTime`) |
| Deduplicação | `dropDuplicatesWithinWatermark` sobre `(order_id, event_sequence)`, watermark de 4 dias |
| Escrita | `MERGE` que só troca o status se `event_sequence` for **maior** que a gravada (idempotente) |
| Tabela | `olist.gold.gold_orders_live`, uma linha por pedido (tabela Delta própria, fora do pipeline `olist.gold`) |

**Parâmetro**

| Widget | Padrão | Uso |
|---|---|---|
| `run_id` | vazio | identificador da execução; no job recebe `{{job.run_id}}`, vazio gera um UUID |

**Partes** (o arquivo é entregue completo, mas cresce a cada parte)

1. Configuração, tabelas, leitura em streaming e redução do micro-lote (esta parte)
2. `foreachBatch` + `MERGE`, `ops.streaming_batches` e execução (etapa 5.3)
3. Regra de risco e lista de pedidos em risco (etapa 5.4)
4. Validação e idempotência (etapa 5.4)

In [ ]:
# O QUE FAZ: cria o parâmetro do notebook (caixa de texto no topo da página).
#
# FUNÇÃO:
# - dbutils.widgets.text(nome, valor_padrão, rótulo): cria um widget de texto; o valor é lido
#   depois com dbutils.widgets.get(nome) e sempre chega como string.
dbutils.widgets.text("run_id", "", "ID da execução")

In [ ]:
# O QUE FAZ: define as constantes do notebook: tabelas de origem e de destino, pasta do checkpoint,
#   tamanho do watermark, SLA e a configuração da regra de risco (ADR 0010, 0011 e 0012).
#   Todos os nomes derivam de CATALOG. Nada aqui lê ou grava dados.
#
# RECURSOS:
# - f"...{variavel}...": f-string, monta texto substituindo as variáveis entre chaves.
# - WATERMARK_DELAY: tempo (em tempo do EVENTO, não do relógio) que o Spark espera por dados
#   atrasados antes de esquecer um evento já visto. O simulador atrasa até 3 dias; 4 dá margem.
# - Checkpoint: pasta onde o Spark guarda até onde já leu e o estado da deduplicação. Apagar a
#   pasta faz o stream recomeçar do zero (o MERGE idempotente torna isso seguro, só mais caro).
# - /Volumes/<catálogo>/<schema>/<volume>: caminho de um Volume do Unity Catalog.
# - uuid.uuid4(): gera um identificador aleatório único; str() converte em texto.
# - "a or b": devolve "a" se tiver valor (texto não vazio); senão, "b".
# - str.strip(): remove espaços e quebras de linha do início e do fim do texto.
import uuid

CATALOG = "olist"

EVENTS_TABLE = f"{CATALOG}.bronze.order_status_events"
ORDERS_TABLE = f"{CATALOG}.bronze.orders"          # data estimada de entrega (etapa 5.3)
LIVE_TABLE = f"{CATALOG}.gold.gold_orders_live"
BATCHES_TABLE = f"{CATALOG}.ops.streaming_batches"  # um registro por micro-lote (etapa 5.3)

CHECKPOINT_DIR = f"/Volumes/{CATALOG}/ops/checkpoints/streaming/gold_orders_live"

WATERMARK_DELAY = "4 days"   # ADR 0010
SLA_MINUTES = 15             # ADR 0012

# Regra de risco (ADR 0011): ajustar aqui, não no código da regra. Usada na etapa 5.4.
RISK_CONFIG = {
    "em_risco_dias": 2,
    "status_abertos": ["created", "approved", "invoiced", "processing", "shipped"],
}

RUN_ID = dbutils.widgets.get("run_id").strip() or str(uuid.uuid4())

print(f"run_id:     {RUN_ID}")
print(f"origem:     {EVENTS_TABLE}")
print(f"destino:    {LIVE_TABLE}")
print(f"checkpoint: {CHECKPOINT_DIR}")
print(f"watermark:  {WATERMARK_DELAY} | SLA: {SLA_MINUTES} min")

## Tabelas

`gold_orders_live` guarda **uma linha por pedido**. As colunas de etapa (`created_at` a `delivered_at`) são preenchidas por **qualquer** evento, na ordem em que chegar; já o `status`, a `event_sequence` e o `event_ts` só avançam quando chega um evento com sequência maior. Por isso um `delivered` que chega antes do `shipped` não perde o instante do despacho quando o `shipped` aparecer depois.

`ops.streaming_batches` registra cada micro-lote (volume, eventos ignorados e latência), base dos critérios de aceite de latência e de eventos tardios.

In [ ]:
# O QUE FAZ: cria as duas tabelas se ainda não existirem. Pode ser executada várias vezes: não
#   altera uma tabela que já existe.
#
# RECURSOS SQL:
# - CREATE TABLE IF NOT EXISTS: cria só se a tabela não existir.
# - COMMENT '...': descrição da tabela ou da coluna, que aparece no Catalog Explorer.
# - TIMESTAMP: data/hora com fuso da sessão; BIGINT: inteiro de 64 bits; DOUBLE: decimal.
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {LIVE_TABLE} (
        order_id                 STRING    COMMENT 'Identificador do pedido (chave: uma linha por pedido).',
        status                   STRING    COMMENT 'Status atual: o do evento de maior event_sequence já aplicado.',
        event_sequence           INT       COMMENT 'Sequência do último evento aplicado; só aumenta.',
        event_ts                 TIMESTAMP COMMENT 'Instante (simulado) do último evento aplicado.',
        created_at               TIMESTAMP COMMENT 'Instante do evento created (compra).',
        approved_at              TIMESTAMP COMMENT 'Instante do evento approved.',
        shipped_at               TIMESTAMP COMMENT 'Instante do evento shipped (despacho).',
        delivered_at             TIMESTAMP COMMENT 'Instante do evento delivered (entrega ao cliente).',
        estimated_delivery_date  TIMESTAMP COMMENT 'Data estimada de entrega informada na compra.',
        risco_atraso             STRING    COMMENT 'normal, em_risco ou atrasado; só pedidos abertos (ADR 0011).',
        _updated_at              TIMESTAMP COMMENT 'Momento em que a linha foi gravada pela última vez.',
        _run_id                  STRING    COMMENT 'Execução que gravou a linha pela última vez.'
    )
    COMMENT 'Estado atual de cada pedido, mantido por streaming (MERGE por event_sequence). Visão operacional ao vivo: os KPIs oficiais continuam na gold batch (ADR 0012).'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BATCHES_TABLE} (
        run_id               STRING    COMMENT 'Execução do notebook (job.run_id ou UUID).',
        batch_id             BIGINT    COMMENT 'Número do micro-lote dentro do checkpoint.',
        processed_at         TIMESTAMP COMMENT 'Momento em que o micro-lote foi gravado.',
        events_in            BIGINT    COMMENT 'Eventos que chegaram ao micro-lote depois da deduplicação.',
        orders_touched       BIGINT    COMMENT 'Pedidos distintos no micro-lote.',
        status_updated       BIGINT    COMMENT 'Pedidos cujo status avançou (sequência maior).',
        ignored_old_sequence BIGINT    COMMENT 'Pedidos cujo evento tinha sequência menor ou igual à gravada.',
        late_events          BIGINT    COMMENT 'Eventos com event_ts mais antigo que o watermark (aplicados, nunca descartados).',
        max_event_ts         TIMESTAMP COMMENT 'Maior event_ts do micro-lote (data simulada corrente).',
        max_latency_seconds  DOUBLE    COMMENT 'Maior intervalo entre a chegada do arquivo na landing e a gravação do micro-lote.'
    )
    COMMENT 'Um registro por micro-lote do streaming gold_orders_live: volume, eventos ignorados, tardios e latência (critérios 5 e 7 da Fase 5).'
""")

print("Tabelas prontas:", LIVE_TABLE, "e", BATCHES_TABLE)

## Leitura em streaming e deduplicação

A origem é a **tabela bronze**, lida como stream: a bronze é Delta *append-only*, então cada linha nova (inclusive duplicatas e reenvios de arquivo) aparece como linha nova para o stream. A deduplicação usa o tempo do evento (`event_ts`) como relógio do watermark.

Pontos de atenção (a verificar na etapa 5.3):

- `dropDuplicatesWithinWatermark` lembra cada chave `(order_id, event_sequence)` só até o watermark passar. Um duplicado que chega **depois** disso não é mais reconhecido pela deduplicação; quem o impede de corromper o estado é o `MERGE` por sequência (segunda defesa, ADR 0010).
- É preciso **medir** se o Spark descarta ou deixa passar eventos mais antigos que o watermark. Se descartar, o critério 7 (evento tardio contado, nunca em silêncio) exige contá-los **antes** da deduplicação. O teste está na Parte 2.

In [ ]:
# O QUE FAZ: declara a função que lê o feed de eventos da bronze como stream, tipa as colunas,
#   descarta o lixo (linha corrompida, pedido sem id) e deduplica dentro do watermark.
#
# FUNÇÃO DECLARADA:
# - read_events_stream(): devolve o DataFrame em streaming com as colunas order_id, status,
#   event_sequence (int), event_ts (timestamp) e as colunas de auditoria da bronze. Não executa
#   nada: um stream só começa a rodar quando é gravado com writeStream.start().
#
# FUNÇÕES NATIVAS:
# - spark.readStream.table(tabela): lê uma tabela Delta como fonte de streaming; cada linha
#   acrescentada à tabela vira linha nova no stream.
# - df.where(condição): filtra linhas. Column.isNull() / isNotNull(): verifica nulo.
# - "&" entre colunas: E lógico do Spark (cada lado entre parênteses, por precedência).
# - Column.cast("int"): converte o texto da bronze em inteiro.
# - F.to_timestamp(coluna, formato): texto -> TIMESTAMP. Usamos TIMESTAMP (e não TIMESTAMP_NTZ)
#   porque é o tipo de relógio mais usado pelo watermark.
# - df.withWatermark(coluna, atraso): declara a coluna de tempo do evento e quanto atraso tolerar;
#   o Spark descarta o estado de deduplicação mais antigo que (maior event_ts visto - atraso).
# - df.dropDuplicatesWithinWatermark(colunas): mantém uma linha por combinação das colunas dentro
#   do watermark. Requer o withWatermark antes.
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

TS_FORMAT = "yyyy-MM-dd HH:mm:ss"


def read_events_stream() -> DataFrame:
    """Eventos de status da bronze, tipados e deduplicados dentro do watermark (stream)."""
    return (
        spark.readStream.table(EVENTS_TABLE)
        .where(F.col("_corrupt_record").isNull() & F.col("order_id").isNotNull())
        .select(
            "order_id",
            "status",
            F.col("event_sequence").cast("int").alias("event_sequence"),
            F.to_timestamp(F.col("event_ts"), TS_FORMAT).alias("event_ts"),
            "_ingested_at",
            "_source_file_modified_at",
        )
        .withWatermark("event_ts", WATERMARK_DELAY)
        .dropDuplicatesWithinWatermark(["order_id", "event_sequence"])
    )

In [ ]:
# O QUE FAZ: confere que a leitura é de fato um stream e mostra o schema. Não grava nada.
#
# RECURSOS:
# - df.isStreaming: True se o DataFrame é uma consulta em streaming.
# - df.printSchema(): lista as colunas e os tipos.
preview = read_events_stream()

print(f"isStreaming: {preview.isStreaming}")
preview.printSchema()

## Redução do micro-lote

Um micro-lote pode trazer **vários eventos do mesmo pedido**, em qualquer ordem. Antes do `MERGE`, o lote é reduzido a **uma linha por pedido**: o evento de maior sequência (status atual) e o instante de cada etapa. É uma função pura (DataFrame → DataFrame), testável sem stream.

In [ ]:
# O QUE FAZ: declara a função pura que reduz um micro-lote a uma linha por pedido, e o mapa de
#   status para a coluna de instante da etapa.
#
# FUNÇÃO DECLARADA:
# - reduce_batch(events): devolve uma linha por order_id com: status, event_sequence e event_ts
#   do evento de MAIOR sequência (desempate pelo event_ts mais recente) e created_at, approved_at,
#   shipped_at e delivered_at (o instante de cada etapa que aparece no lote, nulo se não apareceu).
#
# FUNÇÕES NATIVAS:
# - Window.partitionBy(chave).orderBy(...): janela que separa as linhas por pedido e as ordena.
#   F.col(...).desc(): ordem decrescente.
# - F.row_number().over(janela): numera as linhas de cada pedido a partir de 1; a linha 1 é o evento
#   de maior sequência.
# - df.groupBy(chave).agg(...): agrega por pedido.
# - F.max(F.when(condição, valor)): maior valor entre as linhas que atendem à condição (when sem
#   otherwise devolve nulo nas demais, e max ignora nulos). Dá o instante de cada etapa.
# - df.join(outro, on=coluna, how="inner"): junta pelas colunas indicadas.
# - [... for ... in dict.items()]: list comprehension; desempacota chave e valor do dicionário.
STAGE_COLUMNS = {
    "created": "created_at",
    "approved": "approved_at",
    "shipped": "shipped_at",
    "delivered": "delivered_at",
}


def reduce_batch(events: DataFrame) -> DataFrame:
    """Uma linha por pedido: status do evento de maior sequência + instante de cada etapa do lote."""
    newest_first = Window.partitionBy("order_id").orderBy(F.col("event_sequence").desc(), F.col("event_ts").desc())
    latest = (
        events
        .withColumn("_rank", F.row_number().over(newest_first))
        .where(F.col("_rank") == 1)
        .select("order_id", "status", "event_sequence", "event_ts")
    )
    stages = events.groupBy("order_id").agg(*[
        F.max(F.when(F.col("status") == status, F.col("event_ts"))).alias(column)
        for status, column in STAGE_COLUMNS.items()
    ])
    return latest.join(stages, on="order_id", how="inner")

In [ ]:
# O QUE FAZ: testa reduce_batch com 5 eventos montados à mão, sem stream e sem gravar nada:
#   - pedido A: o delivered (seq 4) chega ANTES do shipped (seq 3) e há um duplicado do created;
#   - pedido B: só o created.
#   Esperado: A fica com status delivered e, mesmo assim, com o instante do shipped preenchido.
#
# FUNÇÕES NATIVAS:
# - spark.createDataFrame(lista_de_tuplas, schema_ddl): DataFrame pequeno a partir de dados Python.
# - datetime(ano, mês, dia, hora, minuto): data/hora do Python.
# - df.collect(): traz as linhas ao driver; aceitável aqui só porque o resultado tem 2 linhas.
# - assert condição, mensagem: interrompe com erro se a condição for falsa.
from datetime import datetime

teste = spark.createDataFrame(
    [
        ("A", "delivered", 4, datetime(2028, 8, 25, 10, 0)),   # chegou primeiro, apesar de ser o último passo
        ("A", "shipped",   3, datetime(2028, 8, 23, 9, 0)),
        ("A", "created",   1, datetime(2028, 8, 22, 8, 0)),
        ("A", "created",   1, datetime(2028, 8, 22, 8, 0)),    # duplicado
        ("B", "created",   1, datetime(2028, 8, 22, 12, 0)),
    ],
    "order_id string, status string, event_sequence int, event_ts timestamp",
)

reduzido = {row["order_id"]: row for row in reduce_batch(teste).collect()}

assert len(reduzido) == 2, "esperado: uma linha por pedido"
assert reduzido["A"]["status"] == "delivered" and reduzido["A"]["event_sequence"] == 4, "A deveria estar delivered (seq 4)"
assert reduzido["A"]["shipped_at"] is not None, "A deveria manter o instante do shipped"
assert reduzido["B"]["status"] == "created" and reduzido["B"]["shipped_at"] is None, "B deveria estar created, sem despacho"

display(reduce_batch(teste).orderBy("order_id"))
print("reduce_batch: OK")